# Data Description
- Author: 詹明翰、謝子詮、曾翊涵（共同作者不代表個別成員已完成工作）
- Date first created: 20261126
- Background: Unit2.2正式固定456里共同主表：10碼研究ID、原固定／人口／歷史碼與名、標準名、固定及歷史面積、crosswalk與原幾何品質；全部里保留，不含人口／土地分區／交通設施特徵。 技術交付Unit2.3–2.7，不是正式Dataset A／B1／B2／B3／C或Finding。
- Unit of observation: 一個固定研究行政里。
- Sample period: 2024–2025事故研究；主地理固定456里，歷史113／114年版僅單時點快照，不認證全年每日法定界。面積固定／年版分開，不把AREA當分母。
- Known issues: 事故原datum未認證、4326僅工作假設；固定原幾何保留157對正面積交集、335個全市union孔洞（個別面內環總數1），沒有snap／buffer修補／填洞／去重。全45,130事故含三筆缺座標未補值；原區名／地址不更正，無最近里／地理編碼。糖廍里同規則納入，原糖?／糖蔀名及provisional_match保留。觀測配對0不等於現實無事故，未制定低樣本排除政策。人工DA／PM驗收仍待完成。
- Definition for each variable: 下方完整41欄字典：dtype、單位、來源、NA及品質；保存於schema，正式欄序與實際reader一致。固定／歷史面積用途另列。
- Data path: `data/processed/taipei_village_base.csv`；456列×41欄、345,137 bytes。
- Source data: schema.source_inputs精確列出19個既有必要檔及本次helper之固定SHA；事故觀測只來自Unit2.1正式CSV，來源文件：[Unit2.1事故說明](taipei_crashes_2024_2025_standardized_documentation.ipynb)、[既有crosswalk說明](village_crosswalk_documentation.ipynb)、[歷史里界原始說明](../raw/unit17_historical_boundaries_raw_documentation.ipynb)。本文件獨立讀取保存CSV／schema，不使用處理Notebook記憶體，不重建raw事故版本或重跑Chapter1。
- Processing code: [prepare_taipei_village_base_and_crash_spatial_assignment.ipynb](../code/prepare_taipei_village_base_and_crash_spatial_assignment.ipynb)，SHA `ef8a3d21f38ecb902f9495720b0094363ebd57cf444b837176fe2b815992abc4`；[必要共用helper](../code/unit22_spatial_support.py)，SHA `06af690caf93af87b0f203d354fc894310aee87eca1d9f6f33df6930b13863fc`。
- Role: geographic_base。
- Verification status: 以本文件乾淨kernel實際讀回、完整assertions及execution metadata為準；本輪Codex協助，成員驗收尚待完成。
- SHA-256: `db0a225df3568ddb2a47eab1e1d28843e1594dec37956c57eea926e444968209`。
- Downstream analysis input: `data/processed/taipei_village_base.csv`，與Data path相同。
- Schema / dictionary: [taipei_village_base_schema.json](taipei_village_base_schema.json)，SHA `a7e5e3b4d77ab506736ecc940e26a87df6cace3a738cbdaeef1e835e6bed7070`。

Primary Owner：謝子詮（DE）；Collaborator / Analytical Reviewer：詹明翰（DA）；Project Reviewer：曾翊涵（PM）。Actual Contribution：Codex本地協助；未推定成員修改、DA驗收或PM審查已完成。


In [1]:
from pathlib import Path
import sys,json,hashlib,platform
import pandas as pd
import numpy as np
from IPython.display import display,Markdown
for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents):
    if (p/'AGENTS.md').is_file() and (p/'templates').is_dir():
        PROJECT_ROOT=p;break
else:raise FileNotFoundError('請從研究工作包內執行')
assert Path(sys.executable).resolve()==(PROJECT_ROOT/'.venv/Scripts/python.exe').resolve()
def sha(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
    return h.hexdigest()
assert sha(PROJECT_ROOT/'data/code/unit22_spatial_support.py')=='06af690caf93af87b0f203d354fc894310aee87eca1d9f6f33df6930b13863fc'
sys.path.insert(0,str(PROJECT_ROOT/'data/code'))
from unit22_spatial_support import Context,show,load_csv,validate_saved_pair,STATES,GEOMETRY_STATES
pd.set_option('display.max_columns',None);pd.set_option('display.max_colwidth',110)
print('Python',platform.python_version(),'existing .venv / DSSI Urban Crash; helper SHA256','06af690caf93af87b0f203d354fc894310aee87eca1d9f6f33df6930b13863fc')


Python 3.12.2 existing .venv / DSSI Urban Crash; helper SHA256 06af690caf93af87b0f203d354fc894310aee87eca1d9f6f33df6930b13863fc


## Change for Your Own Data

按固定CSV、schema、producer及helper hash讀取，任何不同即停止。UTF-8-SIG、LF、__MISSING__；原空字串保留，不用預設NA／dtype推論。附件**MISSING**文字不覆蓋實際schema契約。兩資料獨立讀回並按鍵驗證；不執行Context處理流程。

In [2]:
EXPECTED_OUTPUT_HASHES={'data/processed/taipei_village_base.csv': 'db0a225df3568ddb2a47eab1e1d28843e1594dec37956c57eea926e444968209', 'data/processed/taipei_village_base_schema.json': 'a7e5e3b4d77ab506736ecc940e26a87df6cace3a738cbdaeef1e835e6bed7070', 'data/processed/taipei_crash_spatial_assignment_2024_2025.csv': '2abcb5a69c8998c2f8b40c975404b93d629fd725899f74afc991f349d9201ba5', 'data/processed/taipei_crash_spatial_assignment_2024_2025_schema.json': 'd43e0e3bcc1af2b7683d9466ccb4c441c39a6d9ce56f957b0ed5a89c1b487ad3'}
EXPECTED_PRODUCER_SHA='ef8a3d21f38ecb902f9495720b0094363ebd57cf444b837176fe2b815992abc4'
assert sha(PROJECT_ROOT/'data/code/prepare_taipei_village_base_and_crash_spatial_assignment.ipynb')==EXPECTED_PRODUCER_SHA
for relative,expected in EXPECTED_OUTPUT_HASHES.items():assert sha(PROJECT_ROOT/relative)==expected,relative
base_contract=json.loads((PROJECT_ROOT/'data/processed/taipei_village_base_schema.json').read_text(encoding='utf-8'))
assignment_contract=json.loads((PROJECT_ROOT/'data/processed/taipei_crash_spatial_assignment_2024_2025_schema.json').read_text(encoding='utf-8'))
for relative,expected in base_contract['source_inputs'].items():assert sha(PROJECT_ROOT/relative)==expected,relative
assert base_contract['source_inputs']==assignment_contract['source_inputs']
base=load_csv(PROJECT_ROOT/base_contract['dataset_path'],base_contract)
assignment=load_csv(PROJECT_ROOT/assignment_contract['dataset_path'],assignment_contract)
upstream_contract=json.loads((PROJECT_ROOT/'data/processed/taipei_crashes_2024_2025_standardized_schema.json').read_text(encoding='utf-8'))
upstream=load_csv(PROJECT_ROOT/'data/processed/taipei_crashes_2024_2025_standardized.csv',upstream_contract)
contract=base_contract
df=base


## Summary

In [3]:
print({'data_path':contract['dataset_path'],'rows':len(df),'columns':len(df.columns),'bytes':(PROJECT_ROOT/contract['dataset_path']).stat().st_size,'sha256':sha(PROJECT_ROOT/contract['dataset_path']),'role':contract['role']})
show('來源／程式固定版本',pd.DataFrame(list(contract['source_inputs'].items()),columns=['path','sha256']))
show('CSV與CRS契約',pd.DataFrame([{'csv':str(contract['csv']),'crs':str(contract['crs']),'id_namespaces':str(contract['id_namespaces'])}]).T)

{'data_path': 'data/processed/taipei_village_base.csv', 'rows': 456, 'columns': 41, 'bytes': 345137, 'sha256': 'db0a225df3568ddb2a47eab1e1d28843e1594dec37956c57eea926e444968209', 'role': 'geographic_base'}


**來源／程式固定版本**

,path,sha256
0,data/processed/taipei_crashes_2024_2025_standardized.csv,b2ac0704068319b74f9aef2a28ad810d0cef473894a2a5cd23976f3fa7d2c252
1,data/processed/taipei_crashes_2024_2025_standardized_schema.json,c3f064b3572d940489bdc86bb5c49dfaea8a228a3e7addbc8f9544bad324b650
2,data/processed/taipei_crashes_2024_2025_standardized_documentation.ipynb,7d5dd73cc2f33308c7b4cdc63ff47839feb34c35baef830b70114ae5e4cc6e0b
3,data/code/prepare_taipei_crashes_2024_2025.ipynb,c14dbd34bd3d776339e292b820a049b792a30ec49f2a8fc0173440ed6e146a9e
4,data/raw/G97_A_CAVLGE_P.shp,f7fb4813a2283198ca564f5686823c454c5fd832d2f6981a2ee88a94719be3cc
5,data/raw/G97_A_CAVLGE_P.shx,ab7bc5e63229f676d3580b68e98db59b91a2d327c74ca23b0000ca1c568d414e
6,data/raw/G97_A_CAVLGE_P.dbf,2f43f92e84279a45b2512cf453e9b72f706d1ad1402665c946b1a354f75d8577
7,data/raw/G97_A_CAVLGE_P.prj,8dac7d82ea889063f9eaa7df408fcaeb7307d32831ea973b6534d0139a4523c5
8,data/raw/G97_A_CAVLGE_P.cpg,3ad3031f5503a4404af825262ee8232cc04d4ea6683d42c5dd0a2f2a27ac9824
9,data/raw/113村里界.zip,32d7f1d611f91b1c63bea7290dd4db93b919bf380cce92b634fca992c974dd94


,path,sha256
18,data/code/unit17_validation_support.py,1a18dd248daf08074a5ba7ec659a489fe7b17852195259e8084727d4fd6f3080
19,data/code/unit22_spatial_support.py,06af690caf93af87b0f203d354fc894310aee87eca1d9f6f33df6930b13863fc


**CSV與CRS契約**

,0
csv,"{'encoding': 'utf-8-sig', 'missing_sentinel': '__MISSING__', 'line_terminator': 'LF', 'keep_default_na': F..."
crs,"{'fixed_source': 'EPSG:3826', 'history_source': 'EPSG:3824', 'accident_source': 'EPSG:4326 working assumpt..."
id_namespaces,"{'village_id': 'fixed CPTVID 10 digits, string', 'population_code': 'existing crosswalk 11 digits, string'..."


### Sample

保存正式資料的實際前十列，沒有以處理kernel物件代替磁碟輸入。

In [4]:
display(df.head(10))

,village_id,boundary_code,population_code,district_name,boundary_village_name,population_village_name,standard_village_name,match_method,verification_status,district_code_source,village_area_m2,village_area_km2,source_AREA_attribute,source_geometry_crs,geometry_type,geometry_valid,interior_hole_count,positive_overlap_neighbor_count,positive_overlap_union_m2,geometry_quality_flag,fixed_geometry_wkb_sha256,historical_boundary_code_2024,historical_district_code_2024,historical_village_name_2024,historical_area_m2_2024,historical_area_km2_2024,historical_match_status_2024,historical_source_crs_2024,historical_fixed_iou_2024,historical_fixed_symdiff_m2_2024,historical_geometry_wkb_sha256_2024,historical_boundary_code_2025,historical_district_code_2025,historical_village_name_2025,historical_area_m2_2025,historical_area_km2_2025,historical_match_status_2025,historical_source_crs_2025,historical_fixed_iou_2025,historical_fixed_symdiff_m2_2025,historical_geometry_wkb_sha256_2025
0,6300500020,6300500020,63000050020,中正區,龍福里,龍福里,龍福里,district_and_village_exact,verified_exact_match,6300500,217698.82998,0.217699,217699.73657,EPSG:3826,Polygon,True,0,0,0.0,valid_original,5a990fba0fe632a530b7ec0bfbef087885da3df1428416b410e333487b404d29,63000050020,63000050,龍福里,217698.892271,0.217699,one_to_one_existing_population_code_crosswalk,EPSG:3824,0.995291,1027.650473,3f3996995ad51c451ef5679dae798de407c3ede4d1b7d11c5b60aaef3c6c9e97,63000050020,63000050,龍福里,217698.892271,0.217699,one_to_one_existing_population_code_crosswalk,EPSG:3824,0.995291,1027.650473,3f3996995ad51c451ef5679dae798de407c3ede4d1b7d11c5b60aaef3c6c9e97
1,6300700001,6300700001,63000070001,萬華區,福星里,福星里,福星里,district_and_village_exact,verified_exact_match,6300700,240225.182001,0.240225,224540.10144,EPSG:3826,Polygon,True,0,0,0.0,valid_original,71f7bf6768474b70181307d7e94193d8df0bd6d15570c5cfd3bdc226dac5659d,63000070001,63000070,福星里,233897.129543,0.233897,one_to_one_existing_population_code_crosswalk,EPSG:3824,0.901436,24576.909937,f6a3eaeb9e3c510d9d2dccdf3c72bb1e04c63505ca51bc92208284181de25e32,63000070001,63000070,福星里,233897.135755,0.233897,one_to_one_existing_population_code_crosswalk,EPSG:3824,0.901436,24576.916143,16ac02d9e2981d9de5b85c2deabcb20cef45bd8d86486717d54b7f2cbe72dd5d
2,6300300011,6300300011,63000030011,大安區,龍坡里,龍坡里,龍坡里,district_and_village_exact,verified_exact_match,6300300,145214.882178,0.145215,145214.88422,EPSG:3826,Polygon,True,0,0,0.0,valid_original,97f23ac35c134080b30f86f8eef7e594140ad599ba800b3c452c85cdcafcfe66,63000030011,63000030,龍坡里,145214.758662,0.145215,one_to_one_existing_population_code_crosswalk,EPSG:3824,0.993469,951.540052,ce2ea68231f62629225b71cfbb99fcc07cae0633590a93480635d640cf8671fd,63000030011,63000030,龍坡里,145214.758662,0.145215,one_to_one_existing_population_code_crosswalk,EPSG:3824,0.993469,951.540052,ce2ea68231f62629225b71cfbb99fcc07cae0633590a93480635d640cf8671fd
3,6300700004,6300700004,63000070004,萬華區,新起里,新起里,新起里,district_and_village_exact,verified_exact_match,6300700,123623.223571,0.123623,123617.91376,EPSG:3826,Polygon,True,0,0,0.0,valid_original,c93743e82b7ad86b82e9f15eae058fef44d680ce93227ae166ec768cd639cdfa,63000070004,63000070,新起里,126769.092582,0.126769,one_to_one_existing_population_code_crosswalk,EPSG:3824,0.969385,3892.433415,81057ebbf2d423d69c9faf891d2eb7fc640608b8226241a7e9e8b3fddb57d92a,63000070004,63000070,新起里,126769.099259,0.126769,one_to_one_existing_population_code_crosswalk,EPSG:3824,0.969385,3892.440092,8971e747ab54dddcb8a5d9d3a6bc531549fdd72363bd20956cd2721cfa4c10a0
4,6300500022,6300500022,63000050022,中正區,建國里,建國里,建國里,district_and_village_exact,verified_exact_match,6300500,605005.693442,0.605006,605005.69464,EPSG:3826,Polygon,True,0,0,0.0,valid_original,501fec89a0162267f187ed46dcfa5dbcbd76a431134b35112de2c323aa1bd39f,63000050022,63000050,建國里,599047.428084,0.599047,one_to_one_existing_population_code_crosswalk,EPSG:3824,0.987776,7404.651262,437684e9866be2a86981d56f92987d4611fc86ea50bd4aede8068fbd663fce2d,63000050022,63000050,建國里,59904

### Summary Stats

品質與完整性摘要；不作統計推論、正式EDA／Finding或里事故組成比例。缺值分母／不可比較狀態分開，不將NA當False。

In [5]:
dictionary=pd.DataFrame(contract['columns'])
dictionary['NA_N']=[int(df[c].isna().sum()) for c in dictionary.name]
dictionary['empty_string_N']=[int(df[c].eq('').sum()) if str(df[c].dtype)=='string' else 0 for c in dictionary.name]
show('完整欄位字典、單位／dtype／來源／NA／空字串',dictionary,block=12)
numeric=[r['name'] for r in contract['columns'] if r['dtype'] in ['Int64','Float64']]
show('數值摘要（有效N／mean／sd／min／q25／q50／q75／max）',df[numeric].describe(percentiles=[.25,.5,.75]).T.reset_index())


**完整欄位字典、單位／dtype／來源／NA／空字串**

,name,dtype,definition,unit,source,missing_rule,NA_N,empty_string_N
0,village_id,string,共同研究10碼ID，原固定CPTVID；不得用歷史／人口碼取代,not applicable,固定SHP／既有crosswalk,NA不補0；原字串空白保留,0,0
1,boundary_code,string,固定界原10碼CPTVID,not applicable,固定SHP／既有crosswalk,NA不補0；原字串空白保留,0,0
2,population_code,string,既有crosswalk之11碼人口代碼,not applicable,固定SHP／既有crosswalk,NA不補0；原字串空白保留,0,0
3,district_name,string,固定里界／crosswalk確認的行政區名,not applicable,固定SHP／既有crosswalk,NA不補0；原字串空白保留,0,0
4,boundary_village_name,string,固定界原里名，包含糖蔀里原字形,not applicable,固定SHP／既有crosswalk,NA不補0；原字串空白保留,0,0
5,population_village_name,string,人口原里名，包含糖?里原字形,not applicable,固定SHP／既有crosswalk,NA不補0；原字串空白保留,0,0
6,standard_village_name,string,正式分析標準里名，糖廍里正常納入,not applicable,固定SHP／既有crosswalk,NA不補0；原字串空白保留,0,0
7,match_method,string,既有crosswalk方法原值，未重寫,not applicable,固定SHP／既有crosswalk,NA不補0；原字串空白保留,0,0
8,verification_status,string,既有證據狀態，provisional_match不限制糖廍分析資格,not applicable,固定SHP／既有crosswalk,NA不補0；原字串空白保留,0,0
9,district_code_source,string,原固定SHP七碼CPTID；不是原事故區序數字碼，未截取或推定順序,not applicable,固定SHP／既有crosswalk,NA不補0；原字串空白保留,0,0


,name,dtype,definition,unit,source,missing_rule,NA_N,empty_string_N
12,source_AREA_attribute,Float64,原AREA屬性僅追溯；單位／產製口徑未由本次認證，不作主要分母,原欄單位未認證,固定SHP／既有crosswalk,NA不補0；原字串空白保留,0,0
13,source_geometry_crs,string,原固定PRJ支持的CRS,not applicable,固定SHP／既有crosswalk,NA不補0；原字串空白保留,0,0
14,geometry_type,string,固定原幾何類型,not applicable,固定SHP／既有crosswalk,NA不補0；原字串空白保留,0,0
15,geometry_valid,boolean,固定原面有效；未執行修復,not applicable,固定SHP／既有crosswalk,NA不補0；原字串空白保留,0,0
16,interior_hole_count,Int64,原Polygon interior ring數，未填補,個,固定SHP／既有crosswalk,NA不補0；原字串空白保留,0,0
17,positive_overlap_neighbor_count,Int64,具有嚴格正面積交集的其他原里數，不消除交集,里,固定SHP／既有crosswalk,NA不補0；原字串空白保留,0,0
18,positive_overlap_union_m2,Float64,每里與其他里正面積交集聯集，僅品質、不扣原面積,m²,固定SHP／既有crosswalk,NA不補0；原字串空白保留,0,0
19,geometry_quality_flag,string,原有效面之孔洞／正面積重疊診斷，未修補,not applicable,固定SHP／既有crosswalk,NA不補0；原字串空白保留,0,0
20,fixed_geometry_wkb_sha256,string,本次讀取原固定geometry WKB摘要，供版本／不變性核對,not applicable,固定SHP／既有crosswalk,NA不補0；原字串空白保留,0,0
21,historical_boundary_code_2024,string,歷史原11碼VILLCODE，經既有人口碼對照共同10碼ID，非截碼,not applicable,data/raw/113村里界.zip::村里界歷史圖資_113/VILLAGE_NLSC_1131128.shp,NA不補0；原字串空白保留,0,0


,name,dtype,definition,unit,source,missing_rule,NA_N,empty_string_N
24,historical_area_m2_2024,Float64,該年版原面由3824轉3826後面積，歷史地理密度用此分母,m²,data/raw/113村里界.zip::村里界歷史圖資_113/VILLAGE_NLSC_1131128.shp,NA不補0；原字串空白保留,0,0
25,historical_area_km2_2024,Float64,該歷史年版投影面積／1e6，不冒充固定共同分母,km²,data/raw/113村里界.zip::村里界歷史圖資_113/VILLAGE_NLSC_1131128.shp,NA不補0；原字串空白保留,0,0
26,historical_match_status_2024,string,沿已接受crosswalk完整一對一，糖廍既定接受且provisional原狀態不變,not applicable,data/raw/113村里界.zip::村里界歷史圖資_113/VILLAGE_NLSC_1131128.shp,NA不補0；原字串空白保留,0,0
27,historical_source_crs_2024,string,歷史原始PRJ支持3824；計量轉至3826,not applicable,data/raw/113村里界.zip::村里界歷史圖資_113/VILLAGE_NLSC_1131128.shp,NA不補0；原字串空白保留,0,0
28,historical_fixed_iou_2024,Float64,同研究里歷史與固定面之交集／聯集；不認定法定邊界變更,0–1,data/raw/113村里界.zip::村里界歷史圖資_113/VILLAGE_NLSC_1131128.shp,NA不補0；原字串空白保留,0,0
29,historical_fixed_symdiff_m2_2024,Float64,同研究里歷史與固定面之對稱差面積,m²,data/raw/113村里界.zip::村里界歷史圖資_113/VILLAGE_NLSC_1131128.shp,NA不補0；原字串空白保留,0,0
30,historical_geometry_wkb_sha256_2024,string,轉3826後歷史geometry WKB摘要，沒有修復或替換原檔,not applicable,data/raw/113村里界.zip::村里界歷史圖資_113/VILLAGE_NLSC_1131128.shp,NA不補0；原字串空白保留,0,0
31,historical_boundary_code_2025,string,歷史原11碼VILLCODE，經既有人口碼對照共同10碼ID，非截碼,not applicable,data/raw/114村里界.zip::村里界歷史圖資_114/VILLAGE_NLSC_11401031.shp,NA不補0；原字串空白保留,0,0
32,historical_district_code_2025,string,歷史原TOWNCODE，與固定CPTID分開保留,not applicable,data/raw/114村里界.zip::村里界歷史圖資_114/VILLAGE_NLSC_11401031.shp,NA不補0；原字串空白保留,0,0
33,historical_village_name_2025,string,歷史原里名；年版快照不宣稱全年每天有效,not applicable,data/raw/114村里界.zip::村里界歷史圖資_114/VILLAGE_NLSC_11401031.shp,NA不補0；原字串空白保留,0,0


,name,dtype,definition,unit,source,missing_rule,NA_N,empty_string_N
36,historical_match_status_2025,string,沿已接受crosswalk完整一對一，糖廍既定接受且provisional原狀態不變,not applicable,data/raw/114村里界.zip::村里界歷史圖資_114/VILLAGE_NLSC_11401031.shp,NA不補0；原字串空白保留,0,0
37,historical_source_crs_2025,string,歷史原始PRJ支持3824；計量轉至3826,not applicable,data/raw/114村里界.zip::村里界歷史圖資_114/VILLAGE_NLSC_11401031.shp,NA不補0；原字串空白保留,0,0
38,historical_fixed_iou_2025,Float64,同研究里歷史與固定面之交集／聯集；不認定法定邊界變更,0–1,data/raw/114村里界.zip::村里界歷史圖資_114/VILLAGE_NLSC_11401031.shp,NA不補0；原字串空白保留,0,0
39,historical_fixed_symdiff_m2_2025,Float64,同研究里歷史與固定面之對稱差面積,m²,data/raw/114村里界.zip::村里界歷史圖資_114/VILLAGE_NLSC_11401031.shp,NA不補0；原字串空白保留,0,0
40,historical_geometry_wkb_sha256_2025,string,轉3826後歷史geometry WKB摘要，沒有修復或替換原檔,not applicable,data/raw/114村里界.zip::村里界歷史圖資_114/VILLAGE_NLSC_11401031.shp,NA不補0；原字串空白保留,0,0


**數值摘要（有效N／mean／sd／min／q25／q50／q75／max）**

,index,count,mean,std,min,25%,50%,75%,max
0,village_area_m2,456.0,588925.622437,1351749.281249,31743.936391,134540.816233,207945.633937,424797.635354,16324433.787935
1,village_area_km2,456.0,0.588926,1.351749,0.031744,0.134541,0.207946,0.424798,16.324434
2,source_AREA_attribute,456.0,588925.3456,1351135.000678,31743.93631,134640.283548,210070.97093,428535.896548,16326540.39624
3,interior_hole_count,456.0,0.002193,0.046829,0.0,0.0,0.0,0.0,1.0
4,positive_overlap_neighbor_count,456.0,0.688596,1.256002,0.0,0.0,0.0,1.0,6.0
5,positive_overlap_union_m2,456.0,0.005095,0.076098,0.0,0.0,0.0,0.0,1.150362
6,historical_area_m2_2024,456.0,591771.430973,1364700.769417,31918.181847,134640.35452,209283.347086,424330.602369,16717525.639483
7,historical_area_km2_2024,456.0,0.591771,1.364701,0.031918,0.13464,0.209283,0.424331,16.717526
8,historical_fixed_iou_2024,456.0,0.980551,0.02534,0.722085,0.977038,0.989987,0.99369,0.998801
9,historical_fixed_symdiff_m2_2024,456.0,13146.495951,53202.424732,493.230697,1112.976739,2254.515828,7481.339433,926063.728021


In [6]:
show('行政區及crosswalk狀態',base.groupby(['district_name','verification_status']).size().rename('villages').reset_index())
show('固定與歷史面積分布；不是人口密度或事故密度',base[['village_area_m2','village_area_km2','historical_area_m2_2024','historical_area_m2_2025','historical_area_km2_2024','historical_area_km2_2025']].describe().T.reset_index())
show('原始幾何品質；不修補',base.groupby(['geometry_type','geometry_valid','geometry_quality_flag']).size().rename('villages').reset_index())
show('糖廍正常納入，原始名稱／provisional保留',base[base.standard_village_name.eq('糖廍里')])
show('歷史與固定面差最大前十里',base.nlargest(10,'historical_fixed_symdiff_m2_2024')[['village_id','district_name','standard_village_name','village_area_km2','historical_area_km2_2024','historical_area_km2_2025','historical_fixed_iou_2024','historical_fixed_symdiff_m2_2024']])


**行政區及crosswalk狀態**

,district_name,verification_status,villages
0,中山區,verified_exact_match,42
1,中正區,verified_exact_match,31
2,信義區,verified_exact_match,41
3,內湖區,verified_exact_match,39
4,北投區,verified_exact_match,42
5,南港區,verified_exact_match,20
6,士林區,verified_exact_match,51
7,大同區,verified_exact_match,25
8,大安區,verified_exact_match,53
9,文山區,verified_exact_match,43


**固定與歷史面積分布；不是人口密度或事故密度**

,index,count,mean,std,min,25%,50%,75%,max
0,village_area_m2,456.0,588925.622437,1351749.281249,31743.936391,134540.816233,207945.633937,424797.635354,16324433.787935
1,village_area_km2,456.0,0.588926,1.351749,0.031744,0.134541,0.207946,0.424798,16.324434
2,historical_area_m2_2024,456.0,591771.430973,1364700.769417,31918.181847,134640.35452,209283.347086,424330.602369,16717525.639483
3,historical_area_m2_2025,456.0,591771.430973,1364700.76987,31918.184654,134640.35452,209283.347086,424330.602369,16717525.711893
4,historical_area_km2_2024,456.0,0.591771,1.364701,0.031918,0.13464,0.209283,0.424331,16.717526
5,historical_area_km2_2025,456.0,0.591771,1.364701,0.031918,0.13464,0.209283,0.424331,16.717526


**原始幾何品質；不修補**

,geometry_type,geometry_valid,geometry_quality_flag,villages
0,Polygon,True,valid_original,313
1,Polygon,True,valid_original_holes_and_overlap,1
2,Polygon,True,valid_original_overlap,142


**糖廍正常納入，原始名稱／provisional保留**

,village_id,boundary_code,population_code,district_name,boundary_village_name,population_village_name,standard_village_name,match_method,verification_status,district_code_source,village_area_m2,village_area_km2,source_AREA_attribute,source_geometry_crs,geometry_type,geometry_valid,interior_hole_count,positive_overlap_neighbor_count,positive_overlap_union_m2,geometry_quality_flag,fixed_geometry_wkb_sha256,historical_boundary_code_2024,historical_district_code_2024,historical_village_name_2024,historical_area_m2_2024,historical_area_km2_2024,historical_match_status_2024,historical_source_crs_2024,historical_fixed_iou_2024,historical_fixed_symdiff_m2_2024,historical_geometry_wkb_sha256_2024,historical_boundary_code_2025,historical_district_code_2025,historical_village_name_2025,historical_area_m2_2025,historical_area_km2_2025,historical_match_status_2025,historical_source_crs_2025,historical_fixed_iou_2025,historical_fixed_symdiff_m2_2025,historical_geometry_wkb_sha256_2025
449,6300700015,6300700015,63000070015,萬華區,糖蔀里,糖?里,糖廍里,official_population_code_plus_empirical_local_rule,provisional_match,6300700,151472.861033,0.151473,151473.06856,EPSG:3826,Polygon,True,0,0,0.0,valid_original,8c359e0024712fecee4af72ce661ff4ae74a34366316c028539ec3d3c07be631,63000070015,63000070,糖廍里,151472.017372,0.151472,one_to_one_existing_population_code_crosswalk,EPSG:3824,0.993913,924.801902,83f35c7c6f1d8747af1bb9461b89555c2c4abce61f0f9e1ebf54a4ccce49e313,63000070015,63000070,糖廍里,151472.017372,0.151472,one_to_one_existing_population_code_crosswalk,EPSG:3824,0.993913,924.801902,83f35c7c6f1d8747af1bb9461b89555c2c4abce61f0f9e1ebf54a4ccce49e313


**歷史與固定面差最大前十里**

,village_id,district_name,standard_village_name,village_area_km2,historical_area_km2_2024,historical_area_km2_2025,historical_fixed_iou_2024,historical_fixed_symdiff_m2_2024
215,6301100024,士林區,富洲里,2.430589,3.307706,3.307706,0.722085,926063.728021
143,6301200042,北投區,湖田里,16.324434,16.717526,16.717526,0.971207,482636.123767
263,6301100042,士林區,天母里,1.702004,1.879401,1.879401,0.900924,186662.383803
150,6301100032,士林區,東山里,1.667833,1.50723,1.50723,0.901013,165328.415067
270,6301000025,內湖區,內溝里,4.36744,4.333803,4.333803,0.967818,142299.731235
144,6301200041,北投區,大屯里,4.383768,4.329053,4.329053,0.969717,133953.705073
273,6301100049,士林區,溪山里,10.201475,10.223898,10.223898,0.987344,130076.958032
207,6301100023,士林區,福安里,2.543988,2.613648,2.613648,0.952544,125354.200078
35,6300900016,南港區,舊莊里,5.335155,5.365828,5.365828,0.978577,115865.578292
214,6301200038,北投區,關渡里,1.639826,1.687365,1.687365,0.938522,105517.187994


固定面積分母用於S0／S1及B1／B2／B3共同比較。歷史地理事故密度若採S2／S3，使用對應年度historical_area；另用固定面積作共同分母比較時須另命名，不能混為同一變數。本Unit只提供面積，不計正式里事故密度。歷史原11碼透過既有crosswalk連共同10碼ID，不直接截碼。

### Assertions

獨立保存檔驗證：456里／45,130事故、dtype、完整鍵集合、四規格foreign keys／子集／共同有效ID、候選JSON與NA語義、原座標／來源未改，以及與正式事故一對一銜接。

In [7]:
assert validate_saved_pair(base,assignment,upstream)
for table,con in [(base,base_contract),(assignment,assignment_contract)]:
 assert len(table)==con['expected_rows'] and list(table)==[r['name'] for r in con['columns']]
 for field in con['columns']:assert str(table[field['name']].dtype)==field['dtype']
assert assignment.groupby('source_year')[['eligible_s0','eligible_s1','eligible_s2','eligible_s3']].sum().to_dict()=={'eligible_s0':{2024:21920,2025:22292},'eligible_s1':{2024:22351,2025:22732},'eligible_s2':{2024:21931,2025:22302},'eligible_s3':{2024:22350,2025:22730}}
assert assignment.fixed_assignment_status.eq('district_conflict').sum()==869
assert assignment.fixed_assignment_status.eq('address_conflict').sum()==2
assert assignment.fixed_assignment_status.eq('outside_or_hole').sum()==44
assert assignment.geometry_village_or_status_changed.groupby(assignment.source_year).sum().to_dict()=={2024:544,2025:585}
assert assignment.district_conflict_to_consistent.sum()==239 and assignment.new_district_conflict_including_unassigned.sum()==215
assert set(assignment.fixed_assignment_status).issubset(STATES)
assert set(assignment.historical_assignment_status).issubset(STATES)
# Cardinality-only technical handoff, no feature aggregation or serialized result.
joined=upstream[['record_id','source_year']].merge(assignment[['record_id','eligible_s0','village_id_s0','eligible_s1','village_id_s1']],on='record_id',how='left',validate='one_to_one')
assert len(joined)==45130 and joined.eligible_s0.notna().all()
village_join=joined.merge(base[['village_id','standard_village_name','village_area_km2']],left_on='village_id_s0',right_on='village_id',how='left',validate='many_to_one')
assert len(village_join)==45130 and village_join.standard_village_name.notna().sum()==44212
skeleton=pd.MultiIndex.from_product([base.village_id.tolist(),[2024,2025]],names=['village_id','year']).to_frame(index=False)
assert len(skeleton)==912 and not skeleton.duplicated(['village_id','year']).any() and base.village_id.nunique()==456
show('Unit2.3銜接使用範例：原事故與配對一對一，再接主里，不輸出新資料',village_join.head(10),block=10)
print('可形成912 village-year及456兩年共同里骨架；本文件未計正式Dataset A／C比例。')
for relative,expected in EXPECTED_OUTPUT_HASHES.items():assert sha(PROJECT_ROOT/relative)==expected
for relative,expected in contract['source_inputs'].items():assert sha(PROJECT_ROOT/relative)==expected


**Unit2.3銜接使用範例：原事故與配對一對一，再接主里，不輸出新資料**

,record_id,source_year,eligible_s0,village_id_s0,eligible_s1,village_id_s1,village_id,standard_village_name,village_area_km2
0,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r000001,2024,True,6300200006,True,6300200006,6300200006,興雅里,0.248822
1,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r000003,2024,True,6300500031,True,6300500031,6300500031,三愛里,0.215015
2,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r000006,2024,True,6300100024,True,6300100024,6300100024,新聚里,0.14936
3,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r000007,2024,True,6300100025,True,6300100025,6300100025,復盛里,0.148935
4,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r000009,2024,True,6300300039,True,6300300039,6300300039,臥龍里,0.195223
5,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r000011,2024,True,6300100002,True,6300100002,6300100002,莊敬里,1.281723
6,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r000014,2024,True,6300400003,True,6300400003,6300400003,正得里,0.146164
7,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r000016,2024,True,6300500001,True,6300500001,6300500001,水源里,0.410551
8,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r000019,2024,True,6300600009,True,6300600009,6300600009,大有里,0.207423
9,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r000025,2024,True,6301100025,True,6301100025,6301100025,岩山里,0.491043


可形成912 village-year及456兩年共同里骨架；本文件未計正式Dataset A／C比例。


In [8]:
result={'status':'passed','kind':'base','rows':len(df),'columns':len(df.columns),'sha256':'db0a225df3568ddb2a47eab1e1d28843e1594dec37956c57eea926e444968209','schema_sha256':'a7e5e3b4d77ab506736ecc940e26a87df6cace3a738cbdaeef1e835e6bed7070','independent_saved_file_read':True,'all_45130_candidates_retained':True,'all_456_villages_retained':True,'human_acceptance':'Codex協助；DA／PM待審'}
(PROJECT_ROOT/'data/temp/unit22_validation/base_documentation_summary.json').write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
display(pd.DataFrame([result]))

,status,kind,rows,columns,sha256,schema_sha256,independent_saved_file_read,all_45130_candidates_retained,all_456_villages_retained,human_acceptance
0,passed,base,456,41,db0a225df3568ddb2a47eab1e1d28843e1594dec37956c57eea926e444968209,a7e5e3b4d77ab506736ecc940e26a87df6cace3a738cbdaeef1e835e6bed7070,True,True,True,Codex協助；DA／PM待審


技術交付已核對，實際成員貢獻／DA驗收／PM審查另待完成。後續以本文件固定CSV＋schema＋hash讀取；主規格S0，敏感規格不自動升格，也不要求每個模型重跑四遍。事故特徵合併比例由分子／有效分母重算，不平均年度比例；設籍人口與道路供給均非交通暴露。完成本Unit後停止，未執行Unit2.3或其他正式里特徵。